# Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore

# Load Dataset

In [ ]:
df = pd.read_csv("../data/tanzania.csv")
df.head()

# Add Country Column

In [ ]:
df["Country"] = "Tanzania" 

# Create Date Column

In [ ]:
df["DATE"] = pd.to_datetime(
    df["YEAR"] * 1000 + df["DOY"],
    format="%Y%j"
)

# Extract Month

In [ ]:
df["Month"] = df["DATE"].dt.month
df.head()

# Replace Missing Values

In [ ]:
df.replace(-999, np.nan, inplace=True)

# Check Duplicates

In [ ]:
duplicates = df.duplicated().sum()
print("Duplicates:", duplicates)

df = df.drop_duplicates()

# Summary Statistics

In [ ]:
df.describe()

# Missing Value Report

In [ ]:
missing = df.isna().sum()

missing_percent = (df.isna().sum() / len(df)) * 100

missing_report = pd.DataFrame({
    "Missing Count": missing,
    "Missing %": missing_percent
})

missing_report.sort_values("Missing %", ascending=False)

# Outlier Detection

In [ ]:
cols = [
    "T2M",
    "T2M_MAX",
    "T2M_MIN",
    "PRECTOTCORR",
    "RH2M",
    "WS2M",
    "WS2M_MAX"
]

z_scores = np.abs(df[cols].apply(zscore))

outliers = (z_scores > 3).sum()

print(outliers)

# Missing Value Cleaning

In [ ]:
df = df[df.isnull().mean(axis=1) < 0.30]

df.fillna(method="ffill", inplace=True)

# Export Clean Data

In [ ]:
df.to_csv("../data/tanzania_clean.csv", index=False)

# Monthly Temperature Plot

In [ ]:
monthly_temp = df.groupby("Month")["T2M"].mean()

monthly_temp.plot(figsize=(10,5))
plt.title("Monthly Average Temperature")
plt.xlabel("Month")
plt.ylabel("Temperature")
plt.show()

# Monthly Rainfall Plot

In [ ]:
monthly_rain = df.groupby("Month")["PRECTOTCORR"].sum()

monthly_rain.plot(kind="bar", figsize=(10,5))
plt.title("Monthly Rainfall")
plt.show()

# Correlation Heatmap

In [ ]:
plt.figure(figsize=(12,8))
sns.heatmap(df.corr(numeric_only=True), annot=True)
plt.show()

# Scatter Plot 1

In [ ]:
sns.scatterplot(data=df, x="T2M", y="RH2M")
plt.show()

# Scatter Plot 2

In [ ]:
sns.scatterplot(data=df, x="T2M_RANGE", y="WS2M")
plt.show()

# Histogram

In [ ]:
df["PRECTOTCORR"].hist(bins=30)
plt.title("Rainfall Distribution")
plt.show()

# Bubble Chart

In [ ]:
plt.figure(figsize=(10,6))

plt.scatter(
    df["T2M"],
    df["RH2M"],
    s=df["PRECTOTCORR"] * 5,
    alpha=0.5
)

plt.xlabel("Temperature")
plt.ylabel("Humidity")
plt.show()